# task 1

line between nearest readings before/after on the same station, else weekday+hour avg of the 56 train days before, else link mean that day

scored with the organizers' own score_task1 (their lanes, their targets), S_FD from their task 3 code. S_LWR can't be done locally (their docs say so)

out: state.csv + eval csvs

In [ ]:
import bisect
import json
import subprocess
import sys
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd

REL = Path("/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public")
OUT = Path("/kaggle/working")
REPO = Path("/tmp/trafficflowbench-public")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/jacky850/trafficflowbench-public.git", str(REPO)], check=True)
print("organizers repo", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                                        capture_output=True, text=True).stdout.strip())
sys.path.insert(0, str(REPO / "src"))
from task1 import score_task1 as s1
from task3 import score_task3 as s3

RUN_EVAL = True
PANELS = None
HISTORY_DAYS = 56
GRID_EVERY = 7
ALL_DAYS = False
COMPARE = False
EXPECTED_ROWS = {"validation": 3_451_376, "private": 3_284_419}

KEYS = ["timestamp", "station_id", "link_id"]
VALUES = ["speed_kmh", "flow_vph"]
READ_COLS = KEYS + VALUES + ["pct_observed"]
REGIMES = ["R1", "R2", "R3"]

OLD = dict(max_gap=60, max_span=None, joint=True, cross_day=False)
VARIANTS = {
    "old": OLD,
    "separate": dict(OLD, joint=False),
    "separate+xday": dict(OLD, joint=False, cross_day=True),
    "separate+xday gap45": dict(OLD, joint=False, cross_day=True, max_gap=45),
    "separate+xday gap90": dict(OLD, joint=False, cross_day=True, max_gap=90),
    "separate+xday span90": dict(OLD, joint=False, cross_day=True, max_span=90),
}

panels = PANELS or sorted(p.name for p in (REL / "corridors").iterdir() if p.is_dir())
families = {p["corridor_id"]: p["family_id"] for p in json.loads((REPO / "config" / "corridors.json").read_text())["panels"]}
modes = json.loads((REPO / "config" / "task3_lwr_modes.json").read_text())
print("roads:", panels)

In [ ]:
def file_date(path):
    return pd.Timestamp(Path(path).stem[-10:].replace("_", "-"))


@lru_cache(maxsize=4)
def _read(path):
    df = pd.read_parquet(path, columns=READ_COLS)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["station_id"] = df["station_id"].astype(str)
    df["link_id"] = df["link_id"].astype(str)
    return df


def read_day(path):
    return _read(path).copy()


def masked_days(panel, split):
    days = {}
    for regime in REGIMES:
        for f in sorted((REL / "corridors" / panel / split / "mainline_states_masked" / f"mask_regime={regime}").glob("*.parquet")):
            assert file_date(f) not in days, f"{f.name} in two regimes"
            days[file_date(f)] = (regime, f)
    return dict(sorted(days.items()))


def daily_profiles(panel):
    aggs = {}
    for f in sorted((REL / "corridors" / panel / "train" / "mainline_states").rglob("*.parquet")):
        d = pd.read_parquet(f, columns=READ_COLS)
        d["timestamp"] = pd.to_datetime(d["timestamp"], utc=True)
        d["link_id"] = d["link_id"].astype(str)
        ok = d["pct_observed"] >= 75
        d["sv"] = d["speed_kmh"].where(ok)
        d["fv"] = d["flow_vph"].where(ok)
        d["weekday"] = d["timestamp"].dt.dayofweek
        d["hour"] = d["timestamp"].dt.hour
        aggs[file_date(f)] = d.groupby(["link_id", "weekday", "hour"]).agg(
            speed_sum=("sv", "sum"), speed_n=("sv", "count"), flow_sum=("fv", "sum"), flow_n=("fv", "count"))
    return aggs


def hist_before(aggs, date):
    dates = sorted(aggs)
    used = dates[:bisect.bisect_left(dates, date)][-HISTORY_DAYS:]
    if not used:
        return None
    return pd.concat([aggs[d] for d in used]).groupby(level=[0, 1, 2]).sum()


def profile_values(rows, hist):
    n = len(rows)
    if hist is None:
        return np.full(n, np.nan), np.full(n, np.nan)
    h = hist.reset_index()
    keyed = pd.DataFrame({"link_id": rows["link_id"].to_numpy(),
                          "weekday": rows["timestamp"].dt.dayofweek.to_numpy(),
                          "hour": rows["timestamp"].dt.hour.to_numpy()})
    out = []
    for v in ("speed", "flow"):
        by_day = h.assign(m=h[f"{v}_sum"] / h[f"{v}_n"].replace(0, np.nan))[["link_id", "weekday", "hour", "m"]]
        by_hour = h.groupby(["link_id", "hour"])[[f"{v}_sum", f"{v}_n"]].sum()
        by_hour = (by_hour[f"{v}_sum"] / by_hour[f"{v}_n"].replace(0, np.nan)).rename("f").reset_index()
        overall = h[f"{v}_sum"].sum() / max(h[f"{v}_n"].sum(), 1)
        k = keyed.merge(by_day, on=["link_id", "weekday", "hour"], how="left").merge(by_hour, on=["link_id", "hour"], how="left")
        out.append(k["m"].fillna(k["f"]).fillna(overall).to_numpy())
    return out


def straight_line(pv, nv, dp, dn, cfg):
    pv, nv, dp, dn = (np.asarray(a, dtype=float) for a in (pv, nv, dp, dn))
    prev_ok = ~np.isnan(pv) & (dp <= cfg["max_gap"])
    next_ok = ~np.isnan(nv) & (dn <= cfg["max_gap"])
    both = prev_ok & next_ok
    if cfg["max_span"]:
        both &= (dp + dn) <= cfg["max_span"]
    res = np.full(len(pv), np.nan)
    res[prev_ok] = pv[prev_ok]
    res[next_ok & ~prev_ok] = nv[next_ok & ~prev_ok]
    nearer_next = prev_ok & next_ok & ~both & (dn < dp)
    res[nearer_next] = nv[nearer_next]
    res[both] = (pv[both] * dn[both] + nv[both] * dp[both]) / (dp[both] + dn[both])
    return res


def predict_day(day_files, date, targets, hist, cfg):
    dates = [date - pd.Timedelta(days=1), date, date + pd.Timedelta(days=1)] if cfg["cross_day"] else [date]
    df = pd.concat([read_day(day_files[d][1]) for d in dates if d in day_files], ignore_index=True)
    df = df.sort_values(["link_id", "station_id", "timestamp"])
    keys = [df["link_id"], df["station_id"]]
    ok = df["pct_observed"] >= 75
    have = {c: df[c].notna() for c in VALUES}
    both_present = have["speed_kmh"] & have["flow_vph"]
    ctx = df[KEYS].copy()
    for c in VALUES:
        good = ok & (both_present if cfg["joint"] else have[c])
        seen, vals = df["timestamp"].where(good), df[c].where(good)
        ctx["pt_" + c] = seen.groupby(keys).ffill()
        ctx["nt_" + c] = seen.groupby(keys).bfill()
        ctx["pv_" + c] = vals.groupby(keys).ffill()
        ctx["nv_" + c] = vals.groupby(keys).bfill()
    out = targets.merge(ctx, on=KEYS, how="left", validate="one_to_one")
    h_speed, h_flow = profile_values(out, hist)
    result = {}
    for c, h in (("speed_kmh", h_speed), ("flow_vph", h_flow)):
        dp = (out["timestamp"] - out["pt_" + c]).dt.total_seconds().to_numpy() / 60
        dn = (out["nt_" + c] - out["timestamp"]).dt.total_seconds().to_numpy() / 60
        line = straight_line(out["pv_" + c], out["nv_" + c], dp, dn, cfg)
        good = ok & have[c]
        link_mean = out["link_id"].map(df[good].groupby("link_id")[c].mean()).to_numpy()
        filled = pd.Series(line).fillna(pd.Series(h)).fillna(pd.Series(link_mean)).fillna(df.loc[good, c].mean())
        result[c] = filled.to_numpy()
        if c == "speed_kmh":
            result["span_min"] = np.nan_to_num(dp, nan=999) + np.nan_to_num(dn, nan=999)
    return np.clip(result["speed_kmh"], 1, None), np.clip(result["flow_vph"], 0, None), result["span_min"]


def load_sheet(panel, split, dates=None):
    path = REL / "task1" / panel / split / "sample_submission_state.csv"
    parts = []
    for chunk in pd.read_csv(path, dtype=str, chunksize=500_000):
        if dates is not None:
            chunk = chunk[chunk["timestamp"].str[:10].isin(dates)]
        if len(chunk):
            parts.append(chunk)
    sheet = pd.concat(parts, ignore_index=True)
    sheet["ts"] = pd.to_datetime(sheet["timestamp"], utc=True)
    sheet["date"] = sheet["timestamp"].str[:10]
    return sheet


def predict_split(panel, split, cfg, aggs, only_dates=None, hist_fixed=None):
    days = masked_days(panel, split)
    if only_dates is not None:
        days_used = {d: v for d, v in days.items() if d in only_dates}
    else:
        days_used = days
    sheet = load_sheet(panel, split, {d.strftime("%Y-%m-%d") for d in days_used} if only_dates is not None else None)
    pieces = []
    for date, (regime, _) in days_used.items():
        rows = sheet[(sheet["date"] == date.strftime("%Y-%m-%d")) & (sheet["mask_regime"] == regime)]
        if rows.empty:
            continue
        targets = rows[["station_id", "link_id"]].copy()
        targets["timestamp"] = rows["ts"].to_numpy()
        hist = hist_fixed if hist_fixed is not None else hist_before(aggs, date)
        speed, flow, span = predict_day(days, date, targets, hist, cfg)
        piece = rows[["panel", "timestamp", "ts", "station_id", "link_id", "mask_regime"]].copy()
        piece["speed_kmh"], piece["flow_vph"], piece["span_min"] = speed, flow, span
        pieces.append(piece)
    return pd.concat(pieces, ignore_index=True), sheet

In [ ]:
def official_task1(panel, preds, names=None):
    original = s1.masked_files
    if names is not None:
        s1.masked_files = lambda rel, p, split, regime: [f for f in original(rel, p, split, regime) if f.name in names]
    try:
        sub = preds.drop(columns=["timestamp"]).rename(columns={"ts": "timestamp"})
        return pd.DataFrame(s1.score_panel(panel, REL, "train", sub))
    finally:
        s1.masked_files = original


def overall(detail):
    p = detail[detail["level"] == "panel"].assign(family=lambda d: d["panel"].map(families))
    return p.groupby("family")["S_state"].mean().mean()


def sample_days(days):
    return {d for r in REGIMES for d in [d for d, (rr, _) in days.items() if rr == r][::GRID_EVERY]}


def official_fd(panel, preds, names=None):
    sub = preds.drop(columns=["timestamp"]).rename(columns={"ts": "timestamp"})
    original = s3.files
    if names is not None:
        s3.files = lambda panel_dir, split: [f for f in original(panel_dir, split) if f.name in names]
    try:
        rows = []
        for r in REGIMES:
            d, missing = s3.derive_physics(panel, REL, "train", sub, r)
            res = s3.score_panel(panel, d, REL, modes, r)
            rows.append({"panel": panel, "regime": r, "S_FD": res.get("S_FD", 0.0), "missing_targets": missing})
        return pd.DataFrame(rows)
    finally:
        s3.files = original


def error_breakdown(panel, preds):
    truth_files = {f.name: f for f in (REL / "corridors" / panel / "train" / "mainline_states").rglob("*.parquet")}
    days = masked_days(panel, "train")
    lanes_map = s1.station_lanes(REL / "corridors" / panel)
    topo = pd.read_csv(REL / "corridors" / panel / "network" / "lwr_mainline_topology.csv", dtype={"link_id": str})
    free = topo.drop_duplicates("link_id").set_index("link_id")["free_speed_kmh"]
    preds = preds.assign(day=preds["ts"].dt.normalize().dt.tz_localize(None))
    parts = []
    for date, (regime, f) in days.items():
        p = preds[preds["day"] == date]
        if p.empty:
            continue
        t = read_day(truth_files[f.name])[KEYS + VALUES].rename(columns={"speed_kmh": "s_true", "flow_vph": "f_true"})
        m = p.drop(columns=["timestamp"]).rename(columns={"ts": "timestamp"}).merge(t, on=KEYS, how="inner")
        parts.append(m)
    m = pd.concat(parts, ignore_index=True).dropna(subset=["s_true", "f_true"])
    lanes = s1.lane_vector(m["station_id"], m["link_id"], lanes_map)
    m["se_speed"] = (m["speed_kmh"] - m["s_true"]) ** 2
    m["se_flow_lane"] = ((m["flow_vph"] - m["f_true"]) / lanes) ** 2
    m["jam"] = m["s_true"] <= 0.6 * m["link_id"].map(free)
    m["gap"] = pd.cut(m["span_min"], [0, 15, 30, 60, 120, 1e9], labels=["<=15", "15-30", "30-60", "60-120", ">120 or one side"])
    g = m.groupby(["jam", "gap"], observed=True).agg(cells=("se_speed", "size"), mse_speed=("se_speed", "mean"),
                                                     mse_flow_lane=("se_flow_lane", "mean"))
    g["rmse_speed"], g["rmse_flow_lane"] = np.sqrt(g.pop("mse_speed")), np.sqrt(g.pop("mse_flow_lane"))
    return g.reset_index().assign(panel=panel)

variants, every 7th train day per mask level, official scorer (run 1 nov):

old 0.9050, separate speed/flow 0.9050, +day before/after 0.9053, gap45 0.9050, gap90 0.9054, span90 0.9053

all within 0.0004 = noise, so we keep old. COMPARE = True reruns it (~40 min)

In [ ]:
if RUN_EVAL and COMPARE:
    aggs_by_panel = {p: daily_profiles(p) for p in panels}
    grid = []
    for name, cfg in VARIANTS.items():
        details = []
        for panel in panels:
            days = masked_days(panel, "train")
            subset = sample_days(days)
            preds, _ = predict_split(panel, "train", cfg, aggs_by_panel[panel], only_dates=subset)
            det = official_task1(panel, preds, {f.name for d, (_, f) in days.items() if d in subset})
            assert det["n_missing_predictions"].sum() == 0, (name, panel)
            details.append(det)
        det = pd.concat(details, ignore_index=True)
        grid.append({"variant": name, "S_state": overall(det),
                     **det[det["level"] == "panel"].set_index("panel")["S_state"].round(4).to_dict()})
        print(name, round(grid[-1]["S_state"], 4))
    grid = pd.DataFrame(grid).sort_values("S_state", ascending=False)
    grid.to_csv(OUT / "task1_grid_every7days.csv", index=False)
    display(grid)

old method on every 7th train day per mask level (~39 days per road, ALL_DAYS = True for all 273): official S_state per road/regime, S_FD, errors by jam / gap

In [ ]:
if RUN_EVAL:
    if "aggs_by_panel" not in globals():
        aggs_by_panel = {p: daily_profiles(p) for p in panels}
    details, fds, errs = [], [], []
    for panel in panels:
        days = masked_days(panel, "train")
        subset = None if ALL_DAYS else sample_days(days)
        names = None if ALL_DAYS else {f.name for d, (_, f) in days.items() if d in subset}
        preds, _ = predict_split(panel, "train", OLD, aggs_by_panel[panel], only_dates=subset)
        det = official_task1(panel, preds, names)
        assert det["n_missing_predictions"].sum() == 0, panel
        details.append(det)
        try:
            fds.append(official_fd(panel, preds, names))
        except Exception as e:
            print("S_FD failed", panel, repr(e))
        try:
            errs.append(error_breakdown(panel, preds))
        except Exception as e:
            print("error breakdown failed", panel, repr(e))
        print(panel, len(subset or days), "days", round(det[det["level"] == "panel"]["S_state"].iloc[0], 4), flush=True)
        del preds
    table = pd.concat(details, ignore_index=True)
    table.to_csv(OUT / "task1_official_eval.csv", index=False)
    print("overall S_state:", round(overall(table), 4))
    display(table[table["level"] == "panel_regime"][["panel", "regime", "n_target_cells", "rmse_speed_kmh",
                                                       "rmse_flow_vph_per_lane", "S_speed", "S_flow", "S_state"]].round(4))
    if fds:
        fds = pd.concat(fds, ignore_index=True)
        fds.to_csv(OUT / "task1_physics_S_FD.csv", index=False)
        display(fds.round(4))
        print("mean S_FD:", round(fds["S_FD"].mean(), 4))
    if errs:
        errs = pd.concat(errs, ignore_index=True)
        errs.to_csv(OUT / "task1_errors_jam_gap.csv", index=False)
        e = errs.assign(sse_s=errs["rmse_speed"] ** 2 * errs["cells"], sse_f=errs["rmse_flow_lane"] ** 2 * errs["cells"])
        g = e.groupby(["jam", "gap"], observed=True)[["cells", "sse_s", "sse_f"]].sum()
        g["rmse_speed"] = np.sqrt(g.pop("sse_s") / g["cells"])
        g["rmse_flow_lane"] = np.sqrt(g.pop("sse_f") / g["cells"])
        g["share_of_speed_error"] = g["rmse_speed"] ** 2 * g["cells"] / (g["rmse_speed"] ** 2 * g["cells"]).sum()
        display(g.round(3))
BEST = OLD

march + april with the winner

In [ ]:
def build_split(split):
    out_path = OUT / f"state_{split}.csv"
    total, first = 0, True
    for panel in panels:
        aggs = aggs_by_panel[panel] if RUN_EVAL else daily_profiles(panel)
        first_day = min(masked_days(panel, split))
        result, sheet = predict_split(panel, split, BEST, aggs, hist_fixed=hist_before(aggs, first_day))
        k5 = ["panel", "timestamp", "station_id", "link_id", "mask_regime"]
        assert len(result) == len(sheet), (panel, split, len(result), len(sheet))
        assert not result.duplicated(k5).any(), (panel, split)
        assert result.merge(sheet[k5], on=k5, how="outer", indicator=True)["_merge"].eq("both").all(), (panel, split)
        assert np.isfinite(result[VALUES].to_numpy()).all(), (panel, split)
        assert set(result["mask_regime"]) <= set(REGIMES)
        result[k5 + VALUES].to_csv(out_path, mode="w" if first else "a", header=first, index=False)
        first = False
        total += len(result)
        print(split, panel, len(result), sorted(result["mask_regime"].unique()))
    if PANELS is None:
        assert total == EXPECTED_ROWS[split], (split, total)
    return out_path


parts = [build_split(s) for s in ["validation", "private"]]

In [ ]:
with open(OUT / "state.csv", "w") as out:
    for i, p in enumerate(parts):
        with open(p) as f:
            header = f.readline()
            if i == 0:
                out.write(header)
            out.writelines(f)
        p.unlink()

n = sum(1 for _ in open(OUT / "state.csv")) - 1
print("state.csv rows:", f"{n:,}")
if PANELS is None:
    assert n == sum(EXPECTED_ROWS.values()), n